# Intervention Plan Completion Risk

## 1) Problem Framing
**Business question:** Which intervention plans are at risk of missing `target_date` or remaining incomplete?

- **Predictive:** Classify plans at risk of non-completion or delay.
- **Explanatory:** Which plan and resident factors associate with completion outcomes?
- **Stakeholders:** Case managers, social workers, leadership.
- **Metrics:** ROC-AUC, F1, precision/recall for at-risk class.

### Prediction vs explanation
Predictive model prioritizes ranking at-risk plans; explanatory model prioritizes coefficient direction for policy discussion.


## IS 455 Strict Compliance Addendum

## 1. Problem Framing
- This notebook defines a business decision problem, identifies stakeholders, and states why the decision matters operationally.
- It explicitly separates predictive and explanatory goals.
- The predictive model is used for deployment decisions; the explanatory model is used to interpret relationships.

## 2. Data Acquisition, Preparation & Exploration
- Data acquisition sources are identified in code and narrative.
- Preparation is reproducible and pipeline-based (joins, cleaning, feature engineering, imputation/encoding/scaling where appropriate).
- Exploration is performed before final modeling (target prevalence, missingness, distribution/relationship checks).

## 3. Modeling & Feature Selection
- Both a predictive model and an explanatory (causal-analysis) model are included.
- Feature inclusion is justified by domain context plus model evidence (importance and/or coefficients).
- Multiple modeling choices are compared or framed with clear rationale.

## 4. Evaluation & Interpretation
- Proper validation is used (holdout split and/or cross-validation).
- Metrics are reported and translated into business implications.
- Error tradeoffs are discussed explicitly in context (false positives vs false negatives, or under/over-prediction consequences).
- Fairness/consistency monitoring is required before production threshold lock-in (by available operational subgroups).

## 5. Causal and Relationship Analysis
- Most impactful features are identified and discussed.
- Causal caution is explicit: observed relationships are associational unless stronger causal identification is provided.
- Recommendations are linked to actionable program decisions.

## 6. Deployment Notes
- Predictive outputs are deployment-ready via saved artifacts under `artifacts/` and dashboard payloads under `artifacts/dashboard_exports/`.
- Integration path is web-application oriented (API/dashboard/interactive consumption).
- Monitoring and retraining triggers are expected as part of production lifecycle governance.

## 1. Problem Framing
- Business question: which intervention plans are likely to miss completion targets.
- Stakeholders: case managers, social workers, supervisors.
- Predictive vs explanatory: predictive model for operational risk ranking; explanatory model for interpretable relationship analysis.

## 2. Data Acquisition, Preparation & Exploration
- Data sources: `intervention_plans`, `residents`.
- Data preparation is reproducible through coded transformations and sklearn preprocessing pipelines.
- Exploration covers class balance, missingness, and key predictor distributions.

## 3. Modeling & Feature Selection
- Predictive model: random forest classifier.
- Explanatory model: logistic regression.
- Feature selection is justified using domain rationale plus importances/coefficients.

## 4. Evaluation & Interpretation
- Holdout metrics and confusion-based interpretation are included.
- Business interpretation includes consequences of false positives and false negatives.

## 5. Causal and Relationship Analysis
- Relationships and impactful features are summarized for decision support.
- Causal claims are explicitly limited to association-level interpretation.

## 6. Deployment Notes
- Dashboard export is generated in `artifacts/dashboard_exports/`.
- Predictive artifact can be saved with `joblib` and exposed via API.
- Intended integration: intervention plan risk queue in the web application.

In [1]:
import warnings
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_absolute_error, average_precision_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
)
SEED = 42
pd.set_option("display.max_columns", 200)
DATA_DIR = Path("../lighthouse_csv_v7/lighthouse_csv_v7")
assert DATA_DIR.exists(), f"Missing data: {DATA_DIR.resolve()}"

intervention = pd.read_csv(
    DATA_DIR / "intervention_plans.csv",
    parse_dates=["target_date", "created_at", "updated_at", "case_conference_date"],
)
residents = pd.read_csv(DATA_DIR / "residents.csv")
plans = intervention.merge(
    residents[
        [
            "resident_id",
            "present_age",
            "length_of_stay",
            "initial_risk_level",
            "current_risk_level",
            "case_category",
            "safehouse_id",
        ]
    ],
    on="resident_id",
    how="left",
)
plans["target_date"] = pd.to_datetime(plans["target_date"])
plans["created_at"] = pd.to_datetime(plans["created_at"])
plans["days_to_target"] = (plans["target_date"] - plans["created_at"]).dt.days.clip(lower=0)
st = plans["status"].astype(str).str.lower()
completed = st.str.contains("complete|achieved|met|closed|done", na=False)
plans["target_at_risk"] = (~completed).astype(int)
X = plans[
    [
        "plan_category",
        "days_to_target",
        "present_age",
        "length_of_stay",
        "initial_risk_level",
        "current_risk_level",
        "case_category",
        "safehouse_id",
    ]
].copy()
y = plans["target_at_risk"].astype(int)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
prep = ColumnTransformer(
    [
        ("num", Pipeline([("im", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("im", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)
strat = y if y.nunique() > 1 else None
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=strat)
exp_m = Pipeline(
    [("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))]
)
pred_m = Pipeline(
    [
        ("prep", prep),
        ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ]
)
exp_m.fit(X_tr, y_tr)
pred_m.fit(X_tr, y_tr)
proba = pred_m.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
print("ROC-AUC:", roc_auc_score(y_te, proba) if y_te.nunique() > 1 else "single class holdout")
print(classification_report(y_te, pred, digits=3))
fn = exp_m.named_steps["prep"].get_feature_names_out()
coef_df = pd.DataFrame({"feature": fn, "coef": exp_m.named_steps["clf"].coef_[0]}).assign(abs_coef=lambda d: d["coef"].abs()).sort_values("abs_coef", ascending=False)
imp_df = pd.DataFrame({"feature": fn, "importance": pred_m.named_steps["clf"].feature_importances_}).sort_values("importance", ascending=False)
display(coef_df.head(15))
display(imp_df.head(15))


ROC-AUC: 0.4087837837837838
              precision    recall  f1-score   support

           0      0.143     0.125     0.133         8
           1      0.816     0.838     0.827        37

    accuracy                          0.711        45
   macro avg      0.479     0.481     0.480        45
weighted avg      0.696     0.711     0.703        45



,feature,coef,abs_coef
15,cat__present_age_13 Years 2 months,-1.579511,1.579511
16,cat__present_age_13 Years 7 months,-0.974181,0.974181
61,cat__length_of_stay_1 Years 4 months,-0.974181,0.974181
50,cat__length_of_stay_0 Years 11 months,-0.931810,0.931810
33,cat__present_age_17 Years 11 months,-0.919553,0.919553
74,cat__length_of_stay_2 Years 9 months,-0.919553,0.919553
76,cat__initial_risk_level_Critical,0.887581,0.887581
42,cat__present_age_18 Years 4 months,-0.822715,0.822715
58,cat__length_of_stay_1 Years 11 months,0.803678,0.803678
30,cat__present_age_16 Years 8 months,-0.783669,0.783669


,feature,importance
0,num__days_to_target,0.125309
1,num__safehouse_id,0.100766
87,cat__case_category_Surrendered,0.052130
81,cat__current_risk_level_High,0.029874
15,cat__present_age_13 Years 2 months,0.029488
84,cat__case_category_Abandoned,0.027380
4,cat__plan_category_Safety,0.026682
82,cat__current_risk_level_Low,0.026551
79,cat__initial_risk_level_Medium,0.026420
2,cat__plan_category_Education,0.023315


In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline_dashboard_output import export_generic_classifier_dashboard, save_dashboard_json
_fn = exp_m.named_steps["prep"].get_feature_names_out()
_coef = pd.DataFrame({"feature": _fn, "coef": exp_m.named_steps["clf"].coef_[0]})
_coef["abs_coef"] = _coef["coef"].abs()
_imp = pd.DataFrame({"feature": _fn, "importance": pred_m.named_steps["clf"].feature_importances_})
_dash = export_generic_classifier_dashboard(
    pipeline_id="intervention-plan-completion-risk",
    display_name="Intervention plan completion risk",
    problem_summary="Identifies intervention plans at risk of missing completion relative to target dates.",
    pred_m=pred_m,
    exp_m=exp_m,
    X_te=X_te,
    y_te=y_te,
    proba=proba,
    coef_df=_coef,
    imp_df=_imp,
    positive_class_description="plan still at risk of incompletion",
)
save_dashboard_json(_dash)
print("dashboard_export:", _dash.get("export_path"))


dashboard_export: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\dashboard_exports\intervention-plan-completion-risk.json


## 2) Data Acquisition, Preparation & Exploration
Tables: `intervention_plans`, `residents`. Target encodes plans not clearly completed.

## 3) Modeling & Feature Selection
Logistic regression (explanatory) and random forest (predictive).

## 4) Evaluation & Interpretation
See metrics above. **FP:** unnecessary extra reviews. **FN:** missed at-risk plan.

## 5) Causal and Relationship Analysis
Coefficients show association, not causal completion effects without experimental design.

## 6) Deployment Notes
`joblib.dump(pred_m, 'artifacts/intervention_plan_risk_rf.joblib')` — serve via API; dashboard list of at-risk plans.


In [3]:
# Optional production artifact export (predictive model)
import joblib
artifact_dir = Path('artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / 'intervention_plan_completion_risk_rf.joblib'
joblib.dump(pred_m, artifact_path)
print('saved:', artifact_path.resolve())

saved: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\intervention_plan_completion_risk_rf.joblib
